# 03 — TimesFM Foundation Model + Classification Head

Path (contrast with notebook 02):
1. **TimesFM 2.0 (500M)** frozen backbone → per-sensor patch embeddings → trainable MLP head
2. Temperature scaling on a held-out calibration set
3. **PolicyLayer**: expected cost + dwell + hysteresis + abstain

Part B extends this to **missing data** by mapping window gaps onto TimesFM's native `input_padding` tokens (1 = ignore) and retraining the head.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics

from sentinelai.baseline.weak_controls import forward_fill_only
from sentinelai.data.skab import load_all_scenarios
from sentinelai.data.windows import inject_gaps, make_windows, temporal_split
from sentinelai.models.timesfm_detector import (
    embed_windows,
    load_timesfm_backbone,
    predict_logits,
    predict_proba,
    train_detector,
)
from sentinelai.policy.decision import PolicyLayer, CostMatrix
from sentinelai.uncertainty.calibration import (
    calibrate_probs,
    expected_calibration_error,
    fit_temperature,
    reliability_curve,
)

## Part A — Clean data (mirror notebook 02)

Load SKAB, forward-fill gaps, window, split temporally. Extract frozen TimesFM embeddings and train only the classification head.

In [ ]:
df = load_all_scenarios(("valve1", "valve2"))

filled = forward_fill_only(df)
batch = make_windows(filled)
train, cal, test = temporal_split(batch)
print(len(train.labels), len(cal.labels), len(test.labels))

In [ ]:
backbone = load_timesfm_backbone()

train_emb = embed_windows(backbone, train, use_mask=False)
cal_emb = embed_windows(backbone, cal, use_mask=False)
test_emb = embed_windows(backbone, test, use_mask=False)

print("embedding dim:", train_emb.shape[1])

In [ ]:
# -- Option: Run PCA on embeddings and take top-k components --

from sklearn.decomposition import PCA

# Toggle to enable PCA. Set to None to disable, or an integer to pick top-k components.
pca_components = 16  

if pca_components is not None:
    pca = PCA(n_components=pca_components)
    pca_train_emb = pca.fit_transform(train_emb)
    pca_cal_emb = pca.transform(cal_emb)
    pca_test_emb = pca.transform(test_emb)
    print(f"PCA applied: embeddings reduced to {pca_components} dimensions")
else:
    pca = None
    pca_train_emb = train_emb
    pca_cal_emb = cal_emb
    pca_test_emb = test_emb


In [ ]:
%load_ext autoreload
%autoreload 2
from sentinelai.models.timesfm_detector import train_detector

result = train_detector(
    pca_train_emb if pca is not None else train_emb,
    train.labels,
    val_emb=pca_cal_emb if pca is not None else cal_emb,
    val_labels=cal.labels,
    hidden_size=16,
    dropout=0.1,
    normalize=True,
    sensor_pool=None,  # pool 8 sensors -> 1280-dim (vs 10240 concat); set None to concat
    epochs=100,
)
model = result.model
scaler = result.scaler
sensor_pool = result.sensor_pool
print(f"Final train loss: {result.train_losses[-1]:.4f}")
print(f"Final val loss: {result.val_losses[-1]:.4f}")

In [ ]:



plt.semilogy(result.train_losses, marker="o", label="Train Log Loss")
plt.semilogy(result.val_losses, marker="o", label="Validation Log Loss")
plt.title("Train/Validation Log Loss During Training (TimesFM head)")
plt.xlabel("Epoch")
plt.ylabel("Log Loss (log scale)")
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()


In [ ]:
p_raw = predict_proba(model, pca_test_emb, scaler=scaler, sensor_pool=sensor_pool)

f1_raw_score = sklearn.metrics.f1_score(test.labels, p_raw > 0.5)
print(f"F1 raw score: {f1_raw_score:.3f}")

In [ ]:
logits_cal = predict_logits(model, pca_cal_emb, scaler=scaler, sensor_pool=sensor_pool)
T = fit_temperature(logits_cal, cal.labels)
print(f"Temperature T = {T:.3f}")

p_cal = calibrate_probs(
    predict_logits(model, pca_test_emb, scaler=scaler, sensor_pool=sensor_pool), T
)

print(f"ECE raw: {expected_calibration_error(p_raw, test.labels):.3f}")
print(f"ECE calibrated: {expected_calibration_error(p_cal, test.labels):.3f}")

In [ ]:
centers, confs_raw, accs_raw = reliability_curve(p_raw, test.labels)
_, confs_cal, accs_cal = reliability_curve(p_cal, test.labels)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax.plot(confs_raw, accs_raw, "o-", label="Raw (uncalibrated)")
ax.plot(confs_cal, accs_cal, "o-", label="Calibrated")
ax.set_xlabel("Mean confidence")
ax.set_ylabel("Accuracy")
ax.set_title("Reliability diagram — TimesFM head")
ax.legend()
plt.show()

In [ ]:
policy1 = PolicyLayer(cost=CostMatrix(cost_fn=100.0, cost_fp=1))
policy1.reset()
records = []
for i in range(len(test.labels)):
    rec = policy1.decide(
        p_hazard_raw=float(p_raw[i]),
        p_hazard_calibrated=float(p_cal[i]),
        coverage=float(test.coverage[i]),
    )
    records.append(rec)

actions = [r.action for r in records]
print({a: actions.count(a) for a in set(actions)})

true_labels = test.labels
is_alarm1 = [r.action == "alarm" for r in records]
policy_accuracy1 = sklearn.metrics.accuracy_score(true_labels, is_alarm1)
print(f"Policy1 accuracy: {policy_accuracy1:.3f}")

In [ ]:
policy2 = PolicyLayer(cost=CostMatrix(cost_fn=1.0, cost_fp=1))
policy2.reset()
records = []
for i in range(len(test.labels)):
    rec = policy2.decide(
        p_hazard_raw=float(p_raw[i]),
        p_hazard_calibrated=float(p_cal[i]),
        coverage=float(test.coverage[i]),
    )
    records.append(rec)

actions = [r.action for r in records]
print({a: actions.count(a) for a in set(actions)})

is_alarm2 = [r.action == "alarm" for r in records]
policy_accuracy2 = sklearn.metrics.accuracy_score(test.labels, is_alarm2)
print(f"Policy2 accuracy: {policy_accuracy2:.3f}")

## Part B — Missing data via input token masking

Inject sensor gaps, then compare:
- **Naive**: zero-filled gaps treated as real values (`use_mask=False`)
- **Masked**: gaps mapped to TimesFM `input_padding=1` so missing tokens are ignored in attention and normalization

In [ ]:
# Config default gap_prob=0.0 — pass an explicit rate so Part B has missing data to mask.
gapped = inject_gaps(df, gap_prob=0.05)
gap_batch = make_windows(gapped)
gap_train, gap_cal, gap_test = temporal_split(gap_batch)
print(
    "mean coverage:",
    float(gap_test.coverage.mean()),
    "min:",
    float(gap_test.coverage.min()),
    "windows with gaps:",
    int((gap_test.coverage < 1.0).sum()),
)

In [ ]:
# Naive path: gaps appear as zeros, not masked in the backbone
naive_train_emb = embed_windows(backbone, gap_train, use_mask=False)
naive_cal_emb = embed_windows(backbone, gap_cal, use_mask=False)
naive_test_emb = embed_windows(backbone, gap_test, use_mask=False)

naive_result = train_detector(
    naive_train_emb,
    gap_train.labels,
    val_emb=naive_cal_emb,
    val_labels=gap_cal.labels,
    normalize=True,
    sensor_pool="mean",
    epochs=100,
)
naive_model = naive_result.model

In [ ]:
# Masked path: input_padding = 1 - window mask for missing timesteps
masked_train_emb = embed_windows(backbone, gap_train, use_mask=True)
masked_cal_emb = embed_windows(backbone, gap_cal, use_mask=True)
masked_test_emb = embed_windows(backbone, gap_test, use_mask=True)

masked_result = train_detector(
    masked_train_emb,
    gap_train.labels,
    val_emb=masked_cal_emb,
    val_labels=gap_cal.labels,
    normalize=True,
    sensor_pool="mean",
    epochs=100,
)
masked_model = masked_result.model

In [ ]:
def eval_gap_model(model, cal_emb, test_emb, test_labels, scaler=None, sensor_pool=None):
    p_raw = predict_proba(model, test_emb, scaler=scaler, sensor_pool=sensor_pool)
    T = fit_temperature(
        predict_logits(model, cal_emb, scaler=scaler, sensor_pool=sensor_pool), gap_cal.labels
    )
    p_cal = calibrate_probs(
        predict_logits(model, test_emb, scaler=scaler, sensor_pool=sensor_pool), T
    )
    f1 = sklearn.metrics.f1_score(test_labels, p_raw > 0.5)
    ece = expected_calibration_error(p_cal, test_labels)
    return {"f1": f1, "ece": ece, "p_raw": p_raw, "p_cal": p_cal, "T": T}


naive_metrics = eval_gap_model(
    naive_model,
    naive_cal_emb,
    naive_test_emb,
    gap_test.labels,
    scaler=naive_result.scaler,
    sensor_pool=naive_result.sensor_pool,
)
masked_metrics = eval_gap_model(
    masked_model,
    masked_cal_emb,
    masked_test_emb,
    gap_test.labels,
    scaler=masked_result.scaler,
    sensor_pool=masked_result.sensor_pool,
)

print(f"Naive  — F1: {naive_metrics['f1']:.3f}, ECE: {naive_metrics['ece']:.3f}")
print(f"Masked — F1: {masked_metrics['f1']:.3f}, ECE: {masked_metrics['ece']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["F1", "ECE"]
naive_vals = [naive_metrics["f1"], naive_metrics["ece"]]
masked_vals = [masked_metrics["f1"], masked_metrics["ece"]]
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width / 2, naive_vals, width, label="Naive (no mask)")
ax.bar(x + width / 2, masked_vals, width, label="Masked (input_padding)")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title("Gapped data: naive vs masked TimesFM embeddings")
ax.legend()
plt.show()

In [ ]:
policy_gap = PolicyLayer(cost=CostMatrix(cost_fn=2.0, cost_fp=1))
policy_gap.reset()
gap_records = []
for i in range(len(gap_test.labels)):
    gap_records.append(
        policy_gap.decide(
            p_hazard_raw=float(masked_metrics["p_raw"][i]),
            p_hazard_calibrated=float(masked_metrics["p_cal"][i]),
            coverage=float(gap_test.coverage[i]),
        )
    )

gap_actions = [r.action for r in gap_records]
print("Masked policy actions:", {a: gap_actions.count(a) for a in set(gap_actions)})
gap_alarm = [r.action == "alarm" for r in gap_records]
print(
    "Masked policy accuracy:",
    f"{sklearn.metrics.accuracy_score(gap_test.labels, gap_alarm):.3f}",
)